In [ ]:
import pandas as pd

safety_df = pd.read_json(
    "./safety_notice_data/safety_notice_info.json"
)

print(safety_df.shape)
print(safety_df.columns)

In [ ]:
print(safety_df.columns)

In [ ]:
print("총 국가 수:", safety_df["ISO코드"].nunique())

In [ ]:
safety_df["작성일"] = pd.to_datetime(
    safety_df["작성일"],
    errors="coerce"
).dt.date

In [ ]:
print(safety_df["작성일"].head())

In [ ]:
print(safety_df["안전공지레벨"].value_counts(dropna=False))

In [ ]:
print(safety_df["안전공지레벨"].unique())
print(safety_df["안전공지레벨"].value_counts(dropna=False))

In [ ]:
import os

print(os.getcwd())

In [ ]:
import pandas as pd

df = pd.read_json("./mle-01-p1-team5/data/raw/safety_notice_info.json")

print(df.shape)
print(df.columns)

In [ ]:
df = df.drop(columns=["안전공지레벨", "대륙코드"])

print(df.columns)
print(df.shape)

In [ ]:
df = df.drop(columns=["대륙영문명"])

print(df.shape)
print(df.columns)

In [ ]:
df = df.rename(columns={
    "작성일": "안전공지_작성일"
})

print(df.columns)

In [ ]:
print(df["안전공지_작성일"].head(10))

In [ ]:
df["안전공지_작성일"] = pd.to_datetime(
    df["안전공지_작성일"],
    errors="coerce"
).dt.date

print(df["안전공지_작성일"].head(10))

In [ ]:
print("작성일 결측치 수:", df["안전공지_작성일"].isna().sum())

In [ ]:
print(
    df[["공지제목", "공지내용"]].head(10)
)

In [ ]:
print(df.loc[0, "공지내용"])

In [ ]:
import re

def has_text(html):
    if pd.isna(html):
        return False

    # HTML 태그 제거
    text = re.sub(r"<[^>]+>", " ", str(html))

    # 마크다운 형태 URL 제거
    text = re.sub(r"\[https?://.*?\]\(https?://.*?\)", " ", text)

    # 남아있는 URL 제거
    text = re.sub(r"https?://\S+", " ", text)

    # 공백 제거 후 실제 글자가 있는지 확인
    text = re.sub(r"\s+", " ", text).strip()

    return len(text) > 0


def has_image(html):
    if pd.isna(html):
        return False

    return bool(re.search(r"<img\b", str(html), flags=re.IGNORECASE))


df["텍스트존재"] = df["공지내용"].apply(has_text)
df["이미지존재"] = df["공지내용"].apply(has_image)

print(df[["텍스트존재", "이미지존재"]].value_counts())

In [ ]:
image_only = df[
    (~df["텍스트존재"]) &
    (df["이미지존재"])
]

print("이미지만 있는 공지:", len(image_only))

display(
    image_only[
        ["국가명", "안전공지_작성일", "공지제목", "공지내용"]
    ].head(10)
)

In [ ]:
empty_notice = df[
    (~df["텍스트존재"]) &
    (~df["이미지존재"])
]

display(
    empty_notice[
        ["국가명", "안전공지_작성일", "공지제목", "공지내용"]
    ]
)

In [ ]:
text_only = df[
    (df["텍스트존재"] == True) &
    (df["이미지존재"] == False)
]

print("텍스트만 있는 공지:", len(text_only))

display(
    text_only[
        ["국가명", "안전공지_작성일", "공지제목", "공지내용"]
    ].head(10)
)

In [ ]:
print(text_only.iloc[0]["공지내용"])

In [ ]:
def extract_image_urls(html):
    if pd.isna(html):
        return None

    urls = re.findall(
        r'https?://[^\s"\')\]]+\.(?:png|jpg|jpeg|gif|webp)',
        str(html),
        flags=re.IGNORECASE
    )

    # 중복 제거
    urls = list(dict.fromkeys(urls))

    if not urls:
        return None

    # CSV 한 셀에 여러 URL 저장
    return " | ".join(urls)


df["공지내용_이미지"] = df["공지내용"].apply(extract_image_urls)

In [ ]:
print(df["공지내용_이미지"].notna().sum())

display(
    df[
        ["공지제목", "공지내용_이미지"]
    ].head(20)
)

In [ ]:
def extract_text(html):
    if pd.isna(html):
        return None

    text = str(html)

    # HTML 태그 제거
    text = re.sub(r"<[^>]+>", " ", text)

    # 마크다운 형태 링크에서 URL 제거
    text = re.sub(
        r"\[https?://.*?\]\(https?://.*?\)",
        " ",
        text
    )

    # HTML 엔티티 간단 정리
    text = text.replace("&nbsp;", " ")
    text = text.replace("&quot;", '"')
    text = text.replace("&amp;", "&")

    # 공백 정리
    text = re.sub(r"\s+", " ", text).strip()

    if text == "":
        return None

    return text


df["공지내용_텍스트"] = df["공지내용"].apply(extract_text)

In [ ]:
print("텍스트 있는 공지:", df["공지내용_텍스트"].notna().sum())

display(
    df[
        ["공지제목", "공지내용_텍스트"]
    ].head(10)
)

In [ ]:
print(df.loc[0, "공지내용_텍스트"])

In [ ]:
print("전체 공지:", len(df))

print(
    "텍스트 있음:",
    df["공지내용_텍스트"].notna().sum()
)

print(
    "이미지 있음:",
    df["공지내용_이미지"].notna().sum()
)

print(
    "둘 다 없음:",
    (
        df["공지내용_텍스트"].isna()
        & df["공지내용_이미지"].isna()
    ).sum()
)

In [ ]:
empty_rows = df[
    df["공지내용_텍스트"].isna()
    & df["공지내용_이미지"].isna()
]

display(
    empty_rows[
        [
            "국가명",
            "공지내용"
        ]
    ]
)

In [ ]:
print(df.loc[278, "공지내용"])

In [ ]:
def extract_attachment_urls(html):
    if pd.isna(html):
        return None

    # 이미지 + PDF 등 URL 추출
    urls = re.findall(
        r'https?://[^\s"\')\]]+',
        str(html)
    )

    # 이미지 또는 PDF 관련 URL만 남기기
    urls = [
        url for url in urls
        if any(ext in url.lower()
               for ext in [".png", ".jpg", ".jpeg", ".gif", ".webp", ".pdf"])
    ]

    # 중복 제거
    urls = list(dict.fromkeys(urls))

    if not urls:
        return None

    return " | ".join(urls)


df["공지내용_첨부링크"] = df["공지내용"].apply(extract_attachment_urls)

In [ ]:
print(
    "첨부링크 있는 공지:",
    df["공지내용_첨부링크"].notna().sum()
)

In [ ]:
print(df.loc[278, "공지내용_첨부링크"])

In [ ]:
df = df.drop(columns=["공지내용_이미지"])

In [ ]:
print(df.columns)

In [ ]:
print("전체 공지:", len(df))

print(
    "텍스트 있음:",
    df["공지내용_텍스트"].notna().sum()
)

print(
    "첨부링크 있음:",
    df["공지내용_첨부링크"].notna().sum()
)

print(
    "둘 다 있음:",
    (
        df["공지내용_텍스트"].notna()
        & df["공지내용_첨부링크"].notna()
    ).sum()
)

print(
    "둘 다 없음:",
    (
        df["공지내용_텍스트"].isna()
        & df["공지내용_첨부링크"].isna()
    ).sum()
)

In [ ]:
remaining = df[
    df["공지내용_텍스트"].isna()
    & df["공지내용_첨부링크"].isna()
]

display(
    remaining[
        [
            "국가명",
            "안전공지_작성일",
            "공지제목",
            "공지내용"
        ]
    ]
)

In [ ]:
for idx in remaining.index:
    print("=" * 50)
    print("index:", idx)
    print("국가명:", df.loc[idx, "국가명"])
    print("제목:", df.loc[idx, "공지제목"])
    print("원문:", df.loc[idx, "공지내용"])

In [ ]:
def extract_attachment_urls(html):
    if pd.isna(html):
        return None

    urls = re.findall(
        r'https?://[^\s"\')\]]+',
        str(html)
    )

    # 이미지 / PDF / YouTube 링크만 남기기
    urls = [
        url for url in urls
        if (
            any(ext in url.lower()
                for ext in [".png", ".jpg", ".jpeg", ".gif", ".webp", ".pdf"])
            or "youtube.com" in url.lower()
            or "youtu.be" in url.lower()
        )
    ]

    # 중복 제거
    urls = list(dict.fromkeys(urls))

    if not urls:
        return None

    return " | ".join(urls)


df["공지내용_첨부링크"] = df["공지내용"].apply(extract_attachment_urls)

In [ ]:
print(df.loc[944, "공지내용_첨부링크"])
print(df.loc[4332, "공지내용_첨부링크"])

In [ ]:
import re
import html

test_html = df.loc[1860, "공지내용"]

alt_texts = re.findall(
    r'alt=["\'](.*?)["\']',
    test_html,
    flags=re.IGNORECASE
)

alt_texts = [
    html.unescape(text).strip()
    for text in alt_texts
    if text.strip()
]

for text in alt_texts:
    print(text)
    print("-" * 50)

In [ ]:
def add_alt_text(row):
    original_text = row["공지내용_텍스트"]
    raw_html = row["공지내용"]

    if pd.isna(raw_html):
        return original_text

    alt_texts = re.findall(
        r'alt=["\'](.*?)["\']',
        str(raw_html),
        flags=re.IGNORECASE
    )

    alt_texts = [
        html.unescape(x).strip()
        for x in alt_texts
        if x.strip()
    ]

    if not alt_texts:
        return original_text

    alt_text = " ".join(alt_texts)

    if pd.isna(original_text):
        return alt_text

    return original_text + " " + alt_text


df["공지내용_텍스트"] = df.apply(add_alt_text, axis=1)

In [ ]:
print(df.loc[1860, "공지내용_텍스트"])

In [ ]:
df = df.drop(columns=[
    "공지내용",
    "텍스트존재",
    "이미지존재"
], errors="ignore")

In [ ]:
print(df.columns)
print(df.shape)

In [ ]:
short_keywords = [
    # 자연재해 / 기상
    "태풍", "폭우", "호우", "홍수", "폭염", "한파", "강설",
    "폭설", "산불", "지진", "해일", "화산", "분화",
    "기상", "대기질",

    # 감염병 / 보건 상황
    "콜레라", "에볼라", "코로나", "지카", "감염",
    "확진", "감염병",

    # 사건 / 분쟁
    "시위", "집회", "파업", "공격", "테러",
    "반군", "교전", "폭발", "선박사고", "교통사고",

    # 일시적 통제 / 경보
    "교통 통제", "통행 제한", "운항 중단",
    "경보", "주의보", "대피", "훈련"
]

long_keywords = [
    # 출입국 / 행정
    "여권", "비자", "ESTA", "통관", "입국", "출국",
    "입출국", "휴대품", "거주등록", "숙박 신고",

    # 범죄 예방
    "보이스피싱", "로맨스 스캠", "사기",
    "소매치기", "절도 예방", "범죄 예방",
    "스토킹", "괴롭힘", "악성 이메일",

    # 법 / 규정
    "교통법규", "드론", "무단 촬영",
    "신분증", "법규",

    # 일반 안전정보
    "안전수칙", "행동요령", "긴급 연락처",
    "여행 가이드", "여행가이드"
]


def classify_notice(row):
    title = str(row["공지제목"]) if pd.notna(row["공지제목"]) else ""
    content = (
        str(row["공지내용_텍스트"])
        if pd.notna(row["공지내용_텍스트"])
        else ""
    )

    text = title + " " + content

    # 여행경보는 유효기간 판단이 어려우므로 보류
    if "여행경보" in text:
        return "판단보류"

    if any(keyword in text for keyword in short_keywords):
        return "단기"

    if any(keyword in text for keyword in long_keywords):
        return "장기"

    return "판단보류"

In [ ]:
print(df["공지유형"].value_counts())

In [ ]:
display(
    df.loc[
        df["공지유형"] == "판단보류",
        ["공지제목"]
    ].head(50)
)

In [ ]:
display(
    df.loc[
        df["공지유형"] == "단기",
        ["공지제목"]
    ].sample(30, random_state=42)
)

In [ ]:
df["공지유형"] = df.apply(classify_notice, axis=1)

print(df["공지유형"].value_counts())

In [ ]:
print(df.columns)

In [ ]:
print(df.isna().sum())

In [ ]:
missing_country = df[
    df["영문국가명"].isna()
    | df["ISO코드"].isna()
    | df["대륙명"].isna()
]

print("결측 행 수:", len(missing_country))

print(
    missing_country["국가명"]
    .value_counts()
)

In [ ]:
print("완전 중복 행:", df.duplicated().sum())

print(
    "공지 기준 중복:",
    df.duplicated(
        subset=["국가명", "안전공지_작성일", "공지제목"]
    ).sum()
)

In [ ]:
dup_mask = df.duplicated(
    subset=["국가명", "안전공지_작성일", "공지제목"],
    keep=False
)

duplicates = df.loc[
    dup_mask,
    [
        "국가명",
        "안전공지_작성일",
        "공지제목",
        "공지내용_텍스트",
        "공지내용_첨부링크"
    ]
].sort_values(
    ["국가명", "안전공지_작성일", "공지제목"]
)

display(duplicates)

In [ ]:
def clean_escaped_html(text):
    if pd.isna(text):
        return None

    # &lt;p&gt; → <p> 형태로 복원
    text = html.unescape(str(text))

    # 복원된 HTML 태그 제거
    text = re.sub(r"<[^>]+>", " ", text)

    # 공백 정리
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None


df["공지내용_텍스트"] = df["공지내용_텍스트"].apply(clean_escaped_html)

In [ ]:
html_left = df[
    df["공지내용_텍스트"]
    .fillna("")
    .str.contains(r"&lt;|&gt;|<[^>]+>", regex=True)
]

print("HTML 흔적 남은 행:", len(html_left))

In [ ]:
print(df.loc[2694, "공지내용_텍스트"])

In [ ]:
print("최종 데이터 크기:", df.shape)
print("\n최종 컬럼:")
print(df.columns)

In [ ]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "safety_notice_processed.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path)
print("최종 shape:", df.shape)

In [ ]:
print(output_path.resolve())

In [ ]:
output_dir = Path(
    r"C:\Users\Playdata\Desktop\단위 프로젝트\mle-01-p1-team5\data\processed"
)

output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "safety_notice_processed.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path)
print("최종 shape:", df.shape)

In [ ]:
df = df.sort_values(
    by=["국가명", "안전공지_작성일"],
    ascending=[True, False]
).reset_index(drop=True)

In [ ]:
display(
    df[["국가명", "안전공지_작성일", "공지제목"]].head(100)
)

In [ ]:
df.to_csv(
    r"C:\Users\Playdata\Desktop\단위 프로젝트\mle-01-p1-team5\data\processed\safety_notice_processed.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import pandas as pd

df = pd.read_csv(
    "mle-01-p1-team5/data/safety_notice_processed.csv",
    encoding="utf-8-sig"
)

# 날짜형 변환
df["안전공지_작성일"] = pd.to_datetime(
    df["안전공지_작성일"],
    errors="coerce"
)

# 연도 생성
df["연도"] = df["안전공지_작성일"].dt.year

# 연도별 안전공지 건수
year_counts = (
    df.groupby("연도")
      .size()
      .reset_index(name="공지건수")
      .sort_values("연도")
)

print(year_counts)

In [ ]:
# 날짜형으로 변환
df["안전공지_작성일"] = pd.to_datetime(
    df["안전공지_작성일"],
    errors="coerce"
)

# 연도 컬럼 생성
df["연도"] = df["안전공지_작성일"].dt.year

# 연도 + 대륙별 안전공지 개수
continent_year = (
    df.dropna(subset=["대륙명"])
      .groupby(["연도", "대륙명"])
      .size()
      .reset_index(name="공지건수")
      .sort_values(["연도", "대륙명"])
)

display(continent_year)

In [ ]:
print("최소 작성일:", df["안전공지_작성일"].min())
print("최대 작성일:", df["안전공지_작성일"].max())

print(
    df["안전공지_작성일"]
    .dt.year
    .value_counts()
    .sort_index()
)

In [ ]:
df.groupby(
    [df["안전공지_작성일"].dt.year,
     df["안전공지_작성일"].dt.month]
).size()

In [ ]:
# 연-월 컬럼 생성
df["연월"] = df["안전공지_작성일"].dt.to_period("M").astype(str)

# 연월 + 대륙별 공지 건수
monthly_continent = (
    df.dropna(subset=["대륙명"])
      .groupby(["연월", "대륙명"])
      .size()
      .reset_index(name="공지건수")
      .sort_values("연월")
)

display(monthly_continent)

In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
print(sys.executable)

In [ ]:
import plotly.express as px
print("plotly OK")

In [ ]:
import sys
print(sys.executable)

import plotly.express as px
print("plotly OK")

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/safety_notice_processed.csv",
    encoding="utf-8-sig"
)

df["안전공지_작성일"] = pd.to_datetime(df["안전공지_작성일"])

print(df.shape)
print(df.columns)

In [ ]:
df["연월"] = df["안전공지_작성일"].dt.to_period("M").astype(str)

monthly_continent = (
    df.dropna(subset=["대륙명"])
      .groupby(["연월", "대륙명"])
      .size()
      .reset_index(name="공지건수")
      .sort_values("연월")
)

display(monthly_continent.head(20))

In [ ]:
import plotly.express as px

fig = px.line(
    monthly_continent,
    x="연월",
    y="공지건수",
    color="대륙명",
    markers=True,
    title="대륙별 월간 안전공지 추이"
)

fig.update_layout(
    xaxis_title="연월",
    yaxis_title="안전공지 건수",
    legend_title="대륙",
    hovermode="x unified"
)

fig.show()

In [ ]:
monthly_continent["연월표시"] = pd.to_datetime(
    monthly_continent["연월"]
).dt.strftime("%y년 %-m월")

In [ ]:
monthly_continent["연월표시"] = pd.to_datetime(
    monthly_continent["연월"]
).apply(lambda x: f"{str(x.year)[2:]}년 {x.month}월")

In [ ]:
fig = px.line(
    monthly_continent,
    x="연월표시",
    y="공지건수",
    color="대륙명",
    markers=True,
    title="대륙별 월간 안전공지 추이"
)

fig.update_layout(
    xaxis_title="연월",
    yaxis_title="안전공지 건수",
    legend_title="대륙",
    hovermode="x unified"
)

fig.show()

In [ ]:
df["연도"] = df["안전공지_작성일"].dt.year
df["월"] = df["안전공지_작성일"].dt.month

monthly_continent = (
    df.dropna(subset=["대륙명"])
      .groupby(["연도", "월", "대륙명"])
      .size()
      .reset_index(name="공지건수")
)

In [ ]:
selected = monthly_continent[
    monthly_continent["연도"] == 2025
]

In [ ]:
fig = px.line(
    selected,
    x="월",
    y="공지건수",
    color="대륙명",
    markers=True,
    title="2025년 대륙별 월간 안전공지 추이"
)

fig.update_xaxes(
    tickmode="linear",
    dtick=1,
    title="월"
)

fig.update_yaxes(
    title="안전공지 건수"
)

fig.show()

In [ ]:
selected_2026 = monthly_continent[
    monthly_continent["연도"] == 2026
]

fig = px.line(
    selected_2026,
    x="월",
    y="공지건수",
    color="대륙명",
    markers=True,
    title="2026년 대륙별 월간 안전공지 추이"
)

fig.update_xaxes(
    tickmode="linear",
    dtick=1,
    title="월"
)

fig.update_yaxes(
    title="안전공지 건수"
)

fig.update_layout(
    legend_title="대륙",
    hovermode="x unified"
)

fig.show()

In [ ]:
high_points = monthly_continent[
    ((monthly_continent["연도"] == 2025) & (monthly_continent["공지건수"] > 80)) |
    ((monthly_continent["연도"] == 2026) & (monthly_continent["공지건수"] > 200))
].sort_values(["연도", "월", "공지건수"], ascending=[True, True, False])

display(high_points)

In [ ]:
peak_details = []

for _, row in high_points.iterrows():

    temp = df[
        (df["안전공지_작성일"].dt.year == row["연도"]) &
        (df["안전공지_작성일"].dt.month == row["월"]) &
        (df["대륙명"] == row["대륙명"])
    ].copy()

    temp["연도"] = row["연도"]
    temp["월"] = row["월"]

    peak_details.append(temp)

peak_details = pd.concat(peak_details, ignore_index=True)

peak_country = (
    peak_details
    .groupby(["연도", "월", "대륙명", "국가명"])
    .size()
    .reset_index(name="공지건수")
    .sort_values(
        ["연도", "월", "대륙명", "공지건수"],
        ascending=[True, True, True, False]
    )
)

display(peak_country)

In [ ]:
peak_country_top5 = (
    peak_country
    .sort_values(
        ["연도", "월", "대륙명", "공지건수"],
        ascending=[True, True, True, False]
    )
    .groupby(["연도", "월", "대륙명"])
    .head(5)
)

display(peak_country_top5)

In [ ]:
title_summary = (
    peak_details
    .groupby(["연도", "월", "대륙명", "공지제목"])
    .size()
    .reset_index(name="반복건수")
    .sort_values(
        ["연도", "월", "대륙명", "반복건수"],
        ascending=[True, True, True, False]
    )
)

title_top5 = (
    title_summary
    .groupby(["연도", "월", "대륙명"])
    .head(5)
)

display(title_top5)

In [1]:
import pandas as pd

# 1. 데이터 불러오기
df = pd.read_csv(
    "./data/safety_notice_processed.csv",
    encoding="utf-8-sig"
)

# 2. 작성일을 날짜형으로 변환
df["안전공지_작성일"] = pd.to_datetime(
    df["안전공지_작성일"],
    errors="coerce"
)

# 3. 연도 / 월 만들기
df["연도"] = df["안전공지_작성일"].dt.year.astype("Int64")
df["월"] = df["안전공지_작성일"].dt.month.astype("Int64")

# 4. 연도 + 월 + 대륙별 공지건수 집계
monthly_continent = (
    df.dropna(subset=["대륙명", "연도", "월"])
      .groupby(["연도", "월", "대륙명"])
      .size()
      .reset_index(name="공지건수")
)

display(monthly_continent.head(10))

,연도,월,대륙명,공지건수
0,2025,3,미주,5
1,2025,3,아주,7
2,2025,3,아프리카,20
3,2025,3,유럽,22
4,2025,3,중동,3
5,2025,4,미주,9
6,2025,4,아주,19
7,2025,4,아프리카,13
8,2025,4,유럽,28
9,2025,4,중동,2


In [2]:
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------
# 1. 선택 위젯
# -------------------------

year_dropdown = widgets.Dropdown(
    options=["전체", 2025, 2026],
    value="전체",
    description="연도:"
)

continent_dropdown = widgets.Dropdown(
    options=["전체"] + sorted(
        monthly_continent["대륙명"].dropna().unique().tolist()
    ),
    value="전체",
    description="대륙:"
)


# -------------------------
# 2. 그래프 + 상세 데이터
# -------------------------

def draw_dashboard(year, continent):

    chart_df = monthly_continent.copy()

    # 연도 필터
    if year != "전체":
        chart_df = chart_df[
            chart_df["연도"] == year
        ]

    # 대륙 필터
    if continent != "전체":
        chart_df = chart_df[
            chart_df["대륙명"] == continent
        ]

    # 연월 표시
    chart_df["연월표시"] = chart_df.apply(
        lambda x: f"{str(x['연도'])[2:]}년 {x['월']}월",
        axis=1
    )

    # -------------------------
    # 그래프
    # -------------------------

    fig = px.line(
        chart_df,
        x="연월표시",
        y="공지건수",
        color="대륙명",
        markers=True,
        title="월별 대륙별 안전공지 추이"
    )

    fig.update_layout(
        xaxis_title="연월",
        yaxis_title="안전공지 건수",
        legend_title="대륙",
        hovermode="x unified"
    )

    fig.show()

    # -------------------------
    # 상세 데이터
    # -------------------------

    detail_df = df.copy()

    if year != "전체":
        detail_df = detail_df[
            detail_df["연도"] == year
        ]

    if continent != "전체":
        detail_df = detail_df[
            detail_df["대륙명"] == continent
        ]

# -------------------------
# 3. 대시보드 실행
# -------------------------

dashboard = widgets.interactive(
    draw_dashboard,
    year=year_dropdown,
    continent=continent_dropdown
)

display(dashboard)

interactive(children=(Dropdown(description='연도:', options=('전체', 2025, 2026), value='전체'), Dropdown(descriptio…